# World Models — Recurrent World Models Facilitate Policy Evolution (2018)

_Papers · Reinforcement Learning_

**Compress what the agent sees with a VAE (V), predict what happens next with an MDN-RNN (M), then evolve a tiny linear controller (C) — and train that controller entirely inside the model's own hallucinated 'dream'.**

---

This notebook builds the three World-Models pieces — V, M, and C — one step at a time, then trains the controller inside the model's own dream. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch (Colab/CPU)

### Step 1 — Sanity-check the worked MDN / temperature example

Before building anything, we reproduce the lesson's pen-and-paper mixture example. M predicts the next latent as a **mixture of Gaussians**; sampling one means picking a component, then drawing `mean + tau·sigma·eps`. The temperature `tau` both widens each Gaussian and (via `pi_k ∝ pi_k^(1/tau)`) flattens the mixing weights — higher `tau` = a more uncertain dream.

In [ ]:
# torch is preinstalled in Colab. No pip install needed.
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

# Density of a 1-D Gaussian, used to evaluate the worked mixture by hand.
def normal_pdf(x, mu, sigma):
    z = (x - mu) / sigma
    return math.exp(-0.5 * z ** 2) / (sigma * math.sqrt(2 * math.pi))

pi    = [0.7, 0.3]      # mixture weights
mu    = [1.0, -2.0]     # component means
sigma = [0.5, 0.5]      # component standard deviations

# Mixture density at z = 1: weighted sum of the two component densities.
dens = sum(p * normal_pdf(1.0, m, s) for p, m, s in zip(pi, mu, sigma))

eps = 0.4               # a fixed unit-normal draw, so the two samples are comparable

# Sample from component 1 at two temperatures: mean + tau*sigma*eps.
samp_t1  = mu[0] + 1.0 * sigma[0] * eps      # tau = 1.0
samp_t13 = mu[0] + 1.3 * sigma[0] * eps      # tau = 1.3

# Reweight the mixing weights at tau = 1.3 via pi_k ∝ pi_k^(1/tau), then renormalize.
reweighted = [p ** (1 / 1.3) for p in pi]
Z = sum(reweighted)
rew = [w / Z for w in reweighted]

print("worked example:  mixture density at z=1 =", round(dens, 4))
print("  tau=1.0 sample =", round(samp_t1, 4), " tau=1.3 sample =", round(samp_t13, 4))
print("  reweighted pi at tau=1.3 =", [round(r, 4) for r in rew])
# mixture density at z=1 = 0.5585 ; tau=1.0 sample = 1.2 ; tau=1.3 sample = 1.26
# reweighted pi at tau=1.3 = [0.6574, 0.3426]

### Step 2 — Define the toy environment and the three networks (V, M, C)

We need a tiny world to learn. `ToyEnv` is a 1-D state in `[-1, 1]`; the action nudges it and the reward `-x^2` rewards staying near 0. Crucially the nudge **flips sign 30% of the time** — the dynamics are stochastic, which is exactly why M must predict a *mixture* (two possible futures) rather than one. We then define **V** (a tiny VAE that compresses the observation), **M** (an MDN-RNN that predicts the next latent as K Gaussians), and **C** (a single linear layer — the only part trained on reward).

In [ ]:
# --- A toy 1-D environment: state x in [-1,1]; action nudges it; reward = -x^2 ---
# (be near 0). Dynamics are STOCHASTIC: the nudge sometimes flips sign -> two modes,
# which is exactly why M needs a MIXTURE, not a single prediction.
class ToyEnv:
    def __init__(self):
        self.x = 0.0

    def reset(self):
        self.x = float(np.random.uniform(-1, 1))
        return np.array([self.x], np.float32)

    def step(self, a):
        flip = -1.0 if np.random.rand() < 0.3 else 1.0      # stochastic: 30% sign flip
        nudge = 0.3 * flip * a
        noise = np.random.normal(0, 0.05)
        self.x = float(np.clip(self.x + nudge + noise, -1, 1))
        r = -self.x ** 2                                     # reward: stay near 0
        return np.array([self.x], np.float32), r


# --- 1. V: a tiny VAE (obs is 1-D here, so V is small; loss = recon + KL). owner: mod-vae ---
class VAE(nn.Module):
    def __init__(self, obs=1, z=2):
        super().__init__()
        self.enc = nn.Linear(obs, 16)
        self.mu = nn.Linear(16, z)
        self.lv = nn.Linear(16, z)
        self.dec = nn.Sequential(nn.Linear(z, 16), nn.ReLU(), nn.Linear(16, obs))

    def encode(self, x):
        h = torch.relu(self.enc(x))
        return self.mu(h), self.lv(h)

    def forward(self, x):
        mu, lv = self.encode(x)
        std = torch.exp(0.5 * lv)
        z = mu + torch.randn_like(mu) * std                 # reparameterization trick
        return self.dec(z), mu, lv


def train_vae(V, X, epochs=300):
    opt = torch.optim.Adam(V.parameters(), 1e-2)
    for _ in range(epochs):
        xr, mu, lv = V(X)
        recon = F.mse_loss(xr, X)                           # reconstruction term
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())  # KL to a unit Gaussian
        loss = recon + 0.001 * kl
        opt.zero_grad()
        loss.backward()
        opt.step()
    return V


# --- 2. M: an MDN-RNN. Output head = K Gaussians over the NEXT latent z. ---
K, ZD = 5, 2
class MDNRNN(nn.Module):
    def __init__(self, z=ZD, a=1, h=32, k=K):
        super().__init__()
        self.k, self.z, self.h = k, z, h
        self.cell = nn.LSTMCell(z + a, h)
        self.head = nn.Linear(h, k * (1 + 2 * z))           # per component: weight, mean(z), logstd(z)

    def step(self, z, a, hc):
        h, c = self.cell(torch.cat([z, a], -1), hc)
        o = self.head(h)
        logpi = o[:, :self.k]
        mu = o[:, self.k:self.k + self.k * self.z]
        logstd = o[:, self.k + self.k * self.z:]
        mu = mu.view(-1, self.k, self.z)
        logstd = logstd.view(-1, self.k, self.z)
        return logpi, mu, logstd, (h, c)


def mdn_nll(logpi, mu, logstd, target):   # mixture negative log-likelihood of true next z
    target = target.unsqueeze(1)                                    # (B,1,z)
    logprob = -0.5 * (((target - mu) / logstd.exp()) ** 2) - logstd - 0.5 * math.log(2 * math.pi)
    logprob = logprob.sum(-1)                                       # sum over z dims -> (B,K)
    log_mix = F.log_softmax(logpi, -1) + logprob
    return -(torch.logsumexp(log_mix, -1)).mean()


def sample_mdn(logpi, mu, logstd, tau=1.0):   # temperature sampling (lesson formula, line 3)
    logpi = logpi / tau                                            # sharpen/flatten weights
    k = torch.distributions.Categorical(logits=logpi).sample()
    idx = k.view(-1, 1, 1).expand(-1, 1, mu.size(-1))
    m = mu.gather(1, idx).squeeze(1)
    s = logstd.exp().gather(1, idx).squeeze(1)
    return m + tau * s * torch.randn_like(m)                       # widen by tau


# --- 3. C: a SINGLE linear layer  a = W_c [z h] + b_c  (the only reward-trained part). ---
class Controller:
    def __init__(self, z=ZD, h=32):
        self.n_in = z + h
        self.W = np.zeros(self.n_in)
        self.b = 0.0

    @property
    def params(self):
        return np.concatenate([self.W, [self.b]])

    def set(self, p):
        self.W = p[:-1]
        self.b = p[-1]

    def act(self, z, h):
        x = np.concatenate([z, h])
        return float(np.tanh(np.dot(self.W, x) + self.b))

### Step 3 — Train V and M from random rollouts

We collect data by acting randomly in the toy env. First we train **V** to compress the 1-D observations into 2-D latents `z`. Then we encode random `(z_t, a_t)` sequences and train **M** to predict the next latent `z_{t+1}` as a mixture — minimizing the mixture negative log-likelihood. Neither V nor M sees any reward; they only learn what the world *looks like* and how it *moves*.

In [ ]:
# --- Build V, M from random-rollout data ---
env = ToyEnv()
Xs = []
for _ in range(400):
    s = env.reset()
    for _ in range(15):
        Xs.append(s)
        s, _ = env.step(np.random.uniform(-1, 1))

X = torch.tensor(np.array(Xs))
V = train_vae(VAE(), X)
with torch.no_grad():
    ZMU, _ = V.encode(X)

# Train M on encoded (z_t, a_t) -> z_{t+1} sequences.
M = MDNRNN()
optM = torch.optim.Adam(M.parameters(), 5e-3)

seqs = []
for _ in range(200):
    s = env.reset()
    zs, as_ = [], []
    for _ in range(15):
        with torch.no_grad():
            zt, _ = V.encode(torch.tensor(s).unsqueeze(0))
        a = np.random.uniform(-1, 1)
        ns, _ = env.step(a)
        zs.append(zt.squeeze(0).numpy())
        as_.append([a])
        s = ns
    seqs.append((np.array(zs), np.array(as_)))

for ep in range(120):
    loss_tot = 0.0
    for zs, as_ in seqs:
        zt = torch.tensor(zs, dtype=torch.float32)
        at = torch.tensor(as_, dtype=torch.float32)
        hc = (torch.zeros(1, M.h), torch.zeros(1, M.h))
        losses = []
        for t in range(len(zt) - 1):
            logpi, mu, logstd, hc = M.step(zt[t:t+1], at[t:t+1], hc)
            losses.append(mdn_nll(logpi, mu, logstd, zt[t+1:t+2]))
        loss = torch.stack(losses).mean()
        optM.zero_grad()
        loss.backward()
        optM.step()
        loss_tot += loss.item()
    if ep % 40 == 0:
        print(f"  M train epoch {ep:3d}  mixture NLL = {loss_tot/len(seqs):.3f}")

### Step 4 — Evolve the controller C: real env vs inside the dream

Now the payoff. C is tiny, so we train it with a gradient-free **evolution strategy** (a CMA-ES stand-in): sample a population of controllers, keep the elite, repeat. We score C two ways — `evaluate_real` runs it in the actual env, while `evaluate_dream` rolls it **entirely inside M** (sampling next-`z` from the mixture, never touching the real env). The test: a controller trained only in the dream should still score well when finally evaluated in reality — World Models' central claim.

In [ ]:
# --- 4. Evolution-strategy trainer for C (gradient-free; CMA-ES stand-in). ---
def evaluate_real(ctrl_params, episodes=8):
    C = Controller()
    C.set(ctrl_params)
    tot = 0.0
    for _ in range(episodes):
        s = env.reset()
        hc = (torch.zeros(1, M.h), torch.zeros(1, M.h))
        with torch.no_grad():
            z, _ = V.encode(torch.tensor(s).unsqueeze(0))
        z = z.squeeze(0).numpy()
        for _ in range(15):
            a = C.act(z, hc[0].squeeze(0).numpy())
            ns, r = env.step(a)
            tot += r
            with torch.no_grad():
                zt, _ = V.encode(torch.tensor(ns).unsqueeze(0))
                _, _, _, hc = M.step(torch.tensor(z).unsqueeze(0), torch.tensor([[a]]), hc)
            z = zt.squeeze(0).numpy()
    return tot / episodes


def evaluate_dream(ctrl_params, tau=1.15, episodes=8):
    # Roll C ENTIRELY inside M: sample next-z from the mixture; never touch the real env.
    C = Controller()
    C.set(ctrl_params)
    tot = 0.0
    for _ in range(episodes):
        s = env.reset()
        with torch.no_grad():
            z, _ = V.encode(torch.tensor(s).unsqueeze(0))
        hc = (torch.zeros(1, M.h), torch.zeros(1, M.h))
        for _ in range(15):
            a = C.act(z.squeeze(0).numpy(), hc[0].squeeze(0).numpy())
            with torch.no_grad():
                logpi, mu, logstd, hc = M.step(z, torch.tensor([[a]]), hc)
                z = sample_mdn(logpi, mu, logstd, tau=tau)
                xr = V.dec(z)                       # decode dreamed latent to a dreamed obs
            tot += float(-(xr.item()) ** 2)         # imagined reward (same shape as real)
    return tot / episodes


def evolve(fitness_fn, n_params, gens=25, pop=24, elite=6, sigma=0.5):
    mean = np.zeros(n_params)
    for g in range(gens):
        pops = mean + sigma * np.random.randn(pop, n_params)
        scores = np.array([fitness_fn(p) for p in pops])
        order = scores.argsort()[::-1]
        mean = pops[order[:elite]].mean(0)          # recenter on the elite
        if g % 8 == 0:
            print(f"    gen {g:2d}  best fitness = {scores.max():.3f}")
    return mean


n = Controller().n_in + 1
print("\nTrain C in the REAL environment:")
C_real = evolve(evaluate_real, n)
print("Train C inside the DREAM (tau=1.15):")
C_dream = evolve(lambda p: evaluate_dream(p, tau=1.15), n)

real_of_real = evaluate_real(C_real, episodes=30)
real_of_dream = evaluate_real(C_dream, episodes=30)
print(f"\nReal-env score  | C trained in REAL  : {real_of_real:.3f}")
print(f"Real-env score  | C trained in DREAM : {real_of_dream:.3f}")
print("The DREAM-trained controller transfers: scored in the REAL env it lands close to")
print("the controller trained in the real env -- policy search never touched the real env.")
# (Our small toy run, not the paper's CarRacing 906+/-21 / Doom 1092+/-556 numbers.)

## Visualize the data & results

_Does a controller trained ENTIRELY inside the learned model M's dream still work in the REAL environment, and how does the dream temperature τ affect that transfer? We train the tiny linear controller C three ways — in the real env, and inside the dream at a too-low τ and a moderate τ — and score all of them in the real environment._

### Step 5 — How the three comparison bars are produced

The visualization compares C trained three ways — in the real env, in a too-cold dream (`tau=0.10`), and in a moderate dream (`tau=1.15`) — all scored in the **real** env. A too-cold dream lets C exploit M's imperfections (high imagined reward, poor real transfer); a warmer dream is harder to cheat, so it transfers. This sketch records that recipe; the full loop lives in the CODE cell above.

In [ ]:
# Sketch of how the three bars above are produced (full loop in the CODE cell).
# Build V (VAE) + M (MDN-RNN) on toy-env rollouts, then evolve the linear controller C
# three ways and score each in the REAL env:
#
#   C_real  = evolve(evaluate_real)                     # trained in the real env (reference)
#   C_cold  = evolve(lambda p: evaluate_dream(p, 0.10)) # trained in dream, too-cold tau
#   C_warm  = evolve(lambda p: evaluate_dream(p, 1.15)) # trained in dream, moderate tau
#
#   evaluate_real(C_real)   # left  bar: reference
#   evaluate_real(C_cold)   # middle bar: poor transfer (C exploited the cold dream)
#   evaluate_real(C_warm)   # right bar: good transfer, close to the real-trained C
#
# Reward is -x^2 (closer to 0 = better). The dream-at-tau=1.15 controller transfers;
# the dream-at-tau=0.10 one does not -- the paper's temperature lesson (Table 2).
# (Numbers are from our own toy run; the paper reports CarRacing 906+/-21 (Table 1) and
#  Doom dream-transfer 1092+/-556 at tau=1.15 (Table 2), not these toy values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked temperature sample. M's $K=2$ mixture for the next latent is weights
            $\pi = (0.7, 0.3)$, means $\mu = (1.0, -2.0)$, standard deviations $\sigma = (0.5, 0.5)$. (a) At
            $\tau = 1$, if the component draw selects component 1 and the unit-normal draw is
            $\varepsilon = 0.4$, what next latent is sampled? (b) Repeat at $\tau = 1.3$. (c) Reweight the
            mixture weights at $\tau = 1.3$ via $\pi_k \propto \pi_k^{1/\tau}$ and renormalize.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- (a) Sample from component 1: $z = \mu_1 + \tau\sigma_1\varepsilon = 1.0 + (1)(0.5)(0.4) = 1.20$. — _Sampling a Gaussian = mean plus (temperature-scaled) std times a unit-normal draw; at $\tau=1$ the width is unchanged._
- (b) At $\tau = 1.3$: $z = 1.0 + (1.3)(0.5)(0.4) = 1.26$. — _Raising $\tau$ scales the component width $\sigma_k \to \tau\sigma_k$, so the same draw lands farther from the mean — a more uncertain dream._
- (c) Reweight: $0.7^{1/1.3} = 0.7^{0.769} = 0.760$, $0.3^{0.769} = 0.396$; sum $= 1.156$; normalized $\pi = (0.760/1.156,\,0.396/1.156) = (0.657,\,0.343)$. — _$\pi_k \propto \pi_k^{1/\tau}$ with $\tau \gt 1$ flattens the weights (the big one shrinks, the small one grows), so higher temperature also spreads probability across modes._

**Answer:** (a) $z = 1.20$; (b) $z = 1.26$; (c) reweighted $\pi = (0.657, 0.343)$. Higher temperature both
                 widens each Gaussian and flattens the mixing weights, making M's dreamed futures more stochastic
                 and uncertain. The notebook recomputes $1.0 + 0.5\cdot0.4 = 1.20$, $1.0 + 1.3\cdot0.5\cdot0.4 =
                 1.26$, and the reweighted $(0.657, 0.343)$.

</details>

**Problem 2.** The ablation — real env vs dream. You have V, M, and C built on the toy environment. Train the
            controller C two ways with everything else identical: (1) by evolution in the real
            environment, and (2) by evolution entirely inside M's dream (C never sees the real env during
            training). Score both in the real environment. What do you expect, and what does the result
            demonstrate about World Models' central claim?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Train C-real: each CMA-ES/ES candidate is scored by running it in the real environment; keep the elite. — _This is the conventional baseline — the policy is optimized directly on real interactions._
- Train C-dream: each candidate is scored by rolling it forward inside M (sample next-$z$ from the mixture at temperature $\tau$), summing imagined reward; never touch the real env during search. Then evaluate the winner in the real env. — _This is "training inside a dream" (§4): if M is a good enough dynamics model, a policy good in the dream is good in reality — and we never paid for real interactions during policy search._
- Compare real-env scores; also try a too-low $\tau$ to see C exploit the dream (high dream score, poor real score). — _The transfer gap, and how $\tau$ controls it, is exactly the paper's Table 2 phenomenon — a hotter dream is harder to cheat, so it transfers better._

**Answer:** The dream-trained controller transfers: scored in the real environment it reaches a
                 reward close to the controller that was trained in the real environment — even though policy
                 search never touched the real env. With the temperature too low, C exploits M's imperfections
                 (high imagined reward, weak real reward); a moderate $\tau$ fixes this. This demonstrates World
                 Models' central claim: once you have a good learned dynamics model M, you can train the policy
                 "in imagination" and carry it back to reality. The CODEVIZ panel shows this contrast on our toy
                 env.

</details>

**Problem 3.** Why does M output a mixture of Gaussians for the next latent instead of a single predicted
            vector $z_{t+1}$? Give the failure mode of the single-vector version in a stochastic environment.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- A single-vector (regression) head trained by mean-squared error learns the conditional mean of the next latent. — _MSE is minimized by the mean; when the future has several distinct outcomes, the mean is a point between them._
- In a stochastic env (the road can curve left OR right), the mean of "left" and "right" is "straight" — an outcome that never actually happens. — _Averaging distinct modes produces a blurry, physically wrong prediction; rolling it forward gives nonsense dreams._
- A mixture of Gaussians keeps the modes separate (one bell at "left", one at "right") and samples one, so dreamed rollouts stay realistic. — _This is precisely why Mixture Density Networks exist and why M is an MDN-RNN (§2.2)._

**Answer:** Because the environment is stochastic, a single predicted $z_{t+1}$ (MSE regression) collapses
                 to the average of the possible futures — e.g. predicting "go straight" when the road will
                 actually curve left or right — producing blurry, unusable dreams. A mixture of Gaussians
                 keeps the distinct futures as separate modes and samples one, so M's hallucinated rollouts stay
                 realistic. That is the whole point of the MDN head on the RNN (§2.2).

</details>